# RUL TRAINING

Unlike the notebooks in the "learning_notebooks" folder - this file is supposed to be a polished final approach to the TRAINING part of the RUL prediction problem, whilst the other notebook act as a small documentation of my learning process.

For this notebook, I decided to use FD004, meaning the dataset that has:
1. 6 unknown operating conditions for the turbines, which we will identify by clustering the data into 6 clusters, using k-means-clustering.
2. 2 fault types instead of just one.

### Imports

In [120]:
import pandas as pd
import matplotlib as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans 
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
import joblib


from features import get_new_features
import preprocessing

import importlib
importlib.reload(preprocessing)

<module 'preprocessing' from '/Users/linus/Documents/Data Projects/01_Predictive_Maintenance/preprocessing.py'>

### Setting up parameters

In [121]:
bundle = {}

sensors = [f"sensor_{i}" for i in range(1,22)]
bundle["sensors"] = sensors

settings = [f"setting_{i}" for i in range(1,4)]
bundle["settings"] = settings

### Loading the training data and RUL

In [122]:
train_data = preprocessing.load_data("./CMAPSSData/train_FD004.txt", sensors=sensors)

# The training data always records reading for an engine till failure. 
# Therefore we can calc the Lifetime for each engine AND the RUL by using the max:

lifetime = train_data.groupby("engine_id")["cycle"].transform("max")
train_data["RUL"] = lifetime - train_data["cycle"]

### Splitting up the training data into the 6 operating conditions
We know from the readme.txt that there is 6 operating conditions. Since for each operating conditions sensors will have different standard / reference conditions, we need to split the training set into these 6 groups

In [123]:
# We need to scale the settings data since we will use k-means clustering for unsupervised learning,
# which is not good at handling numbers in different scales

settings_scaler = StandardScaler()
settings_scaler.fit(train_data[settings])
bundle["settings_scaler"] = settings_scaler

scaled_settings = settings_scaler.transform(train_data[settings])

k_means = KMeans(n_clusters=6, n_init=20, random_state=8)
k_means.fit(scaled_settings)
bundle["k_means"] = k_means

train_data["regime"] = k_means.predict(scaled_settings)

mean = train_data.groupby("regime")[sensors].mean()
std = train_data.groupby("regime")[sensors].std()

### Filtering out minimal variance sensors

In [124]:
useless_sensors = std.columns[std.mean(axis=0) <= 10e-5].to_list()
bundle["useless_sensors"] = useless_sensors

sensors = [s for s in sensors if s not in useless_sensors]
bundle["sensors"] = sensors

train_data = train_data.drop(columns=useless_sensors)

### Calculating reference conditions (healthy conditions) for each sensor in each regime
For this I assume that the first 20 cycles of an engines lifetime are healthy and therefore show reference conditions that I can compare later sensor readings with. 

In [125]:
early_training_data = train_data[train_data["cycle"] <= 20]
reference_mean = early_training_data.groupby("regime")[sensors].mean()
reference_std = early_training_data.groupby("regime")[sensors].std()

### Normalizing sensor data
To identify deviations from the healthy reference values for each sensor, we need to normalize sensor data, in each regime seperately

In [126]:
sensor_scalers = {}

for regime in range(6):
    
    regime_rows = early_training_data["regime"] == regime # Thats a boolean mask for early regime rows
    early_regime_rows = early_training_data.loc[regime_rows, sensors]

    sensor_scaler = StandardScaler()
    sensor_scaler.fit(early_regime_rows)

    sensor_scalers[regime] = sensor_scaler

bundle["sensor_scalers"] = sensor_scalers

train_data = preprocessing.apply_preprocessing(train_data, bundle)

### Adding features

rolling mean, rolling standard deviation, slope, exponential_mean, deviation from ewm, deviation from initial reference

In [127]:
new_features = get_new_features(train_data, sensors)
train_data = pd.concat([train_data, new_features], axis=1)

### Finding best model parameters + training the model
Since the histogram gradient boosting performed best in Notebook 7, we will go off that, and evaluate performance with different parameters using RandomizedSearchCV

it autmatically trains via refit = true

In [128]:
X = train_data[sensors + new_features.columns.tolist() + settings + ["cycle"]]
Y = train_data["RUL"]

model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=8,
    n_jobs=4
)

parameter_options = {
    "n_estimators": [50, 100, 200, 400, 600, 1000],
    "learning_rate": [0.02, 0.035, 0.05, 0.1],
    "max_depth": [2, 3, 4, 5, 8],
    "min_child_weight": [5, 15, 30, 60],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "reg_lambda": [1, 10, 20],
    "reg_alpha": [0, 1, 5]
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=parameter_options,
    n_iter=40,
    scoring="neg_root_mean_squared_error",
    cv=GroupKFold(n_splits=4),
    random_state=8,
    n_jobs=-1, # using all available cpus
    verbose=2,
    refit=True
)

search.fit(X, Y, groups=train_data["engine_id"])
model = search.best_estimator_

print("Best parameters:", search.best_params_)
print("CV RMSE:", -search.best_score_)

Fitting 4 folds for each of 40 candidates, totalling 160 fits
[CV] END colsample_bytree=1.0, learning_rate=0.02, max_depth=2, min_child_weight=5, n_estimators=50, reg_alpha=5, reg_lambda=1, subsample=1.0; total time=   2.9s
[CV] END colsample_bytree=1.0, learning_rate=0.02, max_depth=2, min_child_weight=5, n_estimators=50, reg_alpha=5, reg_lambda=1, subsample=1.0; total time=   2.9s
[CV] END colsample_bytree=1.0, learning_rate=0.02, max_depth=2, min_child_weight=5, n_estimators=50, reg_alpha=5, reg_lambda=1, subsample=1.0; total time=   2.8s
[CV] END colsample_bytree=1.0, learning_rate=0.02, max_depth=2, min_child_weight=5, n_estimators=50, reg_alpha=5, reg_lambda=1, subsample=1.0; total time=   2.8s
[CV] END colsample_bytree=0.7, learning_rate=0.035, max_depth=4, min_child_weight=5, n_estimators=50, reg_alpha=1, reg_lambda=1, subsample=1.0; total time=   3.9s
[CV] END colsample_bytree=0.7, learning_rate=0.035, max_depth=4, min_child_weight=5, n_estimators=50, reg_alpha=1, reg_lambda=1

### Saving data, preprocessing objects and fitted models

In [129]:
bundle["model"] = model
bundle["model_columns"] = X.columns.tolist()

joblib.dump(bundle, "models.joblib")

['models.joblib']